# Preprocessing of Hourly Data

## Libraries

In [24]:
import pandas as pd
import os
from utils import read_files
import datetime

## Traffic Data

In [83]:
# this folder is not on GitHub, because it's too large
path = "data/traffic/"

files_traffic = os.listdir(path)

for i in range(len(files_traffic)):
    files_traffic[i] = os.path.join(path, files_traffic[i])

df_traffic = read_files(file_paths=files_traffic, drop_columns=["MSID", "MSName", "ZSID", "Achse", "HNr", "Hoehe", "EKoord", "NKoord", "Knummer", "Kname", "AnzDetektoren", "D1ID", "D2ID", "D3ID", "D4ID", "LieferDat", "AnzFahrzeugeStatus"], date_column="MessungDatZeit", UTC=False)

Create a map for the stations, that the stations of the traffic data can be merged with the stations of the pollutant and meteo data.

In [66]:
station_map = {
    "Rosengartenstrasse (Nordstrasse)": "Zch_Rosengartenstrasse",
    "Stampfenbachstrasse (Wasserwerkstrasse)": "Zch_Stampfenbachstrasse",
    "Schimmelstrasse (Werdstrasse)": "Zch_Schimmelstrasse",
    "Zürichbergstrasse (Dreiwiesenstrasse)": "Zch_Heubeeribüel"
}

In [ ]:
df_traffic = df_traffic[df_traffic["ZSName"].isin(station_map)]

In [75]:
df_traffic["Standort"] = df_traffic["ZSName"].map(station_map)

In [76]:
# select the necessary columns only
df_traffic = df_traffic.loc[:, ["Year", "Datetime", "Standort", "AnzFahrzeuge"]]

In [ ]:
df_traffic.head()

Check the hourly traffic data for zero values.

In [78]:
print("df_traffic shape:", df_traffic.shape)

print(df_traffic.isnull().sum())
#df_traffic["AnzFahrzeuge"] = df_traffic["AnzFahrzeuge"].astype(int)

df_traffic_year = (
    df_traffic.groupby(["Year", "Standort"], as_index=False)["AnzFahrzeuge"]
    .sum()
)

df_traffic_year


df_traffic shape: (768624, 4)
Year                0
Datetime            0
Standort            0
AnzFahrzeuge    54362
dtype: int64


,Year,Standort,AnzFahrzeuge
0,2012,Zch_Rosengartenstrasse,14584615.0
1,2012,Zch_Schimmelstrasse,9764644.0
2,2012,Zch_Stampfenbachstrasse,4952538.0
3,2013,Zch_Rosengartenstrasse,12529021.0
4,2013,Zch_Schimmelstrasse,9919183.0
5,2013,Zch_Stampfenbachstrasse,4896925.0
6,2014,Zch_Rosengartenstrasse,0.0
7,2014,Zch_Schimmelstrasse,9939737.0
8,2014,Zch_Stampfenbachstrasse,4659701.0
9,2015,Zch_Rosengartenstrasse,6212429.0


In [ ]:
df_traffic_year = (
    df_traffic.groupby(["Year", "Standort"], as_index=False)["AnzFahrzeuge"]
    .sum()
)

df_traffic_year

It seems like in the year 2014, the station at Rosengartenstrasse was not working.

For further analysis, separate the date time column.

In [79]:
df_traffic["Date"] = pd.to_datetime(df_traffic["Datetime"]).dt.date
df_traffic["Time"] = pd.to_datetime(df_traffic["Datetime"]).dt.time
df_traffic["Month"] = pd.to_datetime(df_traffic["Datetime"]).dt.month
df_traffic["Weekday"] = pd.to_datetime(df_traffic["Datetime"]).dt.weekday

In [80]:
df_traffic.head()

,Year,Datetime,Standort,AnzFahrzeuge,Date,Time,Month,Weekday
614880,2012,2012-01-01 00:00:00,Zch_Rosengartenstrasse,561.0,2012-01-01,00:00:00,1,6
614881,2012,2012-01-01 01:00:00,Zch_Rosengartenstrasse,1223.0,2012-01-01,01:00:00,1,6
614882,2012,2012-01-01 02:00:00,Zch_Rosengartenstrasse,989.0,2012-01-01,02:00:00,1,6
614883,2012,2012-01-01 03:00:00,Zch_Rosengartenstrasse,826.0,2012-01-01,03:00:00,1,6
614884,2012,2012-01-01 04:00:00,Zch_Rosengartenstrasse,823.0,2012-01-01,04:00:00,1,6


In [81]:
# check time during summer time
#df_traffic.loc[df_traffic['Datetime'].dt.date == pd.to_datetime('2024-06-18').date(), 'Datetime']

In [18]:
df_traffic.to_csv("data/traffic_h1_12_24.csv", index=False)